# Thai AI Paper Feed — Phase B Stage 2 (ทางเลือก): เทรนผ่าน Unsloth Studio (no-code UI)

ไฟล์นี้แค่ **ติดตั้ง + เปิด Unsloth Studio** (เซลล์ setup/start ก๊อปมาจาก [notebook ทางการ](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb) ตรงๆ ไม่ได้แก้)

จากนั้น **คุณ config เองในหน้า UI ที่ pop-up ขึ้นมา** — ด้านล่างมี checklist ค่าที่ต้องกรอกสำหรับเคส A และเคส B ให้ตรงกับที่คุยกันไว้ใน `docs/PHASE_B_EXECUTION_PLAN.md`

**ต่างจาก `train_finetune_colab.ipynb`** (ไฟล์สคริปต์อีกอัน) ตรงที่:
- ไฟล์นี้ = คลิกปรับ config เองในหน้าเว็บ ทีละเคส ไม่มี auto grid search
- ไฟล์สคริปต์ = เขียนโค้ด train ล้วน วน grid search หา hyperparameter อัตโนมัติได้

ใช้ไฟล์นี้ถ้าอยากลองเห็น loss curve สด, ปรับ config เองแบบจับต้องได้, ไม่อยากเขียนโค้ด

### Setup: Clone repo and run setup

In [ ]:
!git clone --depth 1 --branch main https://github.com/unslothai/unsloth.git
%cd /content/unsloth
!chmod +x studio/setup.sh && ./studio/setup.sh --local

### Start Unsloth Studio

In [ ]:
import sys
sys.path.insert(0, "/content/unsloth/studio/backend")
from colab import start

# ค่า default: เปิดเป็น iframe ในแท็บนี้เลย — start() จะ block ไว้กันเคอร์เนลตาย ห้ามรันเซลล์อื่นระหว่างนี้
start()

# ถ้าอยากได้ลิงก์แชร์แบบเปิดแท็บแยก ให้คอมเมนต์บรรทัด start() ด้านบนแล้วใช้อันนี้แทน:
# start(cloudflare=True)

---
## ก่อนเริ่ม: อัปโหลด dataset เข้า Studio

Studio รองรับสร้าง dataset จากไฟล์ JSON/JSONL ได้ในตัว (ไม่ต้องแปลง format เอง) — อัปโหลดไฟล์ต่อไปนี้ตอนสร้าง dataset ใน UI:

- `phase-b/data/train.jsonl` (340 แถว) — ใช้เทรน
- `phase-b/data/test.jsonl` (60 แถว) — เก็บไว้ eval/preview เท่านั้น **ห้ามเอาไปเทรน**

โครงแต่ละแถวมี field `system` / `user` / `assistant` อยู่แล้ว ตรงกับ chat format มาตรฐาน — ตอนสร้าง dataset ใน Studio ให้ map field ตามนี้: system → system prompt, user → input, assistant → target output

---
## Checklist config — เคส A (7-8B + QLoRA)

กรอกในหน้า UI ตามนี้:

| ตั้งค่า | ค่าที่ใช้ |
|---|---|
| Model | `scb10x/typhoon2-qwen2.5-7b-instruct` |
| Quantization | 4-bit (QLoRA) |
| LoRA rank (r) | เริ่มที่ 16 — ลองปรับ 8 / 32 เทียบ loss ดู |
| LoRA alpha | เท่ากับ r (เช่น r=16 → alpha=16) |
| Target modules | all linear layers (ปกติ Studio เลือกให้อัตโนมัติ) |
| Max sequence length | 4096 (ลดเหลือ 3072/2048 ถ้า OOM) |
| Batch size | 1 |
| Gradient accumulation | 8 |
| Learning rate | เริ่มที่ 2e-4 — ลองปรับ 1e-4 / 4e-4 เทียบ loss curve |
| Epochs | 2-3 |
| Dataset | `train.jsonl` (ตามที่อัปโหลดข้างบน) |

**วิธีลอง "ปรับเอง":** เทรนสั้นๆ ก่อน (ตัด epoch เหลือ 1 หรือกำหนด max steps ถ้า Studio มีตัวเลือก) ดู loss curve สด ถ้า loss นิ่งเร็วเกินไปหรือไม่ลง → ลอง lr สูงขึ้น/ต่ำลง แล้วค่อยรันเต็ม epoch เมื่อเจอค่าที่โอเค

เทรนเสร็จ → ไปหน้า **Export** ของ Studio → save adapter ไว้ (เลือก LoRA adapter หรือ merged model ตามต้องการ) แล้วเก็บลง Google Drive ก่อนปิด session

---
## Checklist config — เคส B (3-4B + LoRA เต็ม 16-bit)

**สำคัญ:** เคส B ตั้งใจ **ไม่ quantize** (ต่างจากเคส A) เพราะโมเดลเล็กพอ VRAM T4 ไหว — ถ้า Studio ถามเรื่อง quantization ให้เลือก 16-bit / none

| ตั้งค่า | ค่าที่ใช้ |
|---|---|
| Model | `google/gemma-4-E4B-it` (ถ้าโหลดไม่ได้/คุณภาพไทยไม่นิ่ง ลอง `Qwen/Qwen2.5-3B-Instruct` แทน) |
| Quantization | **ไม่ quantize** — โหลดเต็ม 16-bit |
| LoRA rank (r) | เริ่มที่ 16 — ลองปรับ 32 / 64 ได้เพราะ VRAM เหลือเยอะกว่าเคส A |
| LoRA alpha | เท่ากับ r |
| Max sequence length | 4096 |
| Batch size | 2 (เผื่อขยับขึ้นได้ถ้า VRAM เหลือ) |
| Gradient accumulation | 4 |
| Learning rate | เริ่มที่ 2e-4 — ลองปรับเทียบ loss curve เหมือนเคส A |
| Epochs | 2-3 |
| Dataset | `train.jsonl` เดียวกับเคส A |

เทรนเสร็จ → Export adapter เก็บลง Drive เหมือนเคส A

---

หลังได้ adapter ทั้ง 2 เคสแล้ว ไปต่อ **Stage 3** ในแผน (`docs/PHASE_B_EXECUTION_PLAN.md`) — เทียบ base vs เคส A vs เคส B vs Gemini ด้วย automatic checks + LLM-judge